# 03 — Screening Analysis (Sub-sesión 2+3)

Carga `_screening_results.parquet` generado por `scripts/run_screening.py` y produce:
- Top-10 por target
- Comparación contra baseline (L32 v3 catboost)
- Heatmaps de AUC
- Análisis de overfitting
- Reporte ejecutivo `informes/03_screening_report.md`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scripts import config as cfg

pd.set_option('display.max_columns', 60)
plt.rcParams['figure.figsize'] = (12, 6)

BASELINE_14 = 0.8453
BASELINE_30 = 0.7912

## 1. Carga

In [ ]:
results_path = cfg.STUDY_SCREENING / '_screening_results.parquet'
assert results_path.exists(), f'No encontrado: {results_path}. Ejecuta scripts/run_screening.py primero.'

df = pd.read_parquet(results_path)
print(f'Total filas: {len(df)}')
print(f'Configs: {df["config_id"].nunique()}')
print(f'Algoritmos: {df["algorithm"].unique()}')
print(f'Cleanups: {df["cleanup"].unique()}')
print(f'Targets: {df["target"].unique()}')
df.head(3)

## 2. Top 10 por target

In [ ]:
show_cols = ['config_id', 'cutoff', 'spike', 'min_logins', 'cleanup', 'algorithm',
             'test_auc_roc', 'test_auc_pr', 'overfitting_gap']

for target in ['churn_14d', 'churn_30d']:
    print(f'\n=== TOP 10 {target} ===')
    top = df[df['target'] == target].nlargest(10, 'test_auc_roc')[show_cols]
    print(top.to_string(index=False))

## 3. Comparación con baseline

In [ ]:
for target, baseline in [('churn_14d', BASELINE_14), ('churn_30d', BASELINE_30)]:
    sub = df[df['target'] == target].copy()
    sub['delta_vs_baseline'] = sub['test_auc_roc'] - baseline
    n_better = (sub['delta_vs_baseline'] > 0.001).sum()
    n_total = len(sub)
    print(f'\n{target}: {n_better}/{n_total} modelos baten baseline {baseline:.4f}')
    if n_better > 0:
        top = sub.nlargest(5, 'delta_vs_baseline')[
            ['config_id', 'cleanup', 'algorithm', 'test_auc_roc', 'delta_vs_baseline']
        ]
        print(top.to_string(index=False))

## 4. Heatmaps de AUC promedio

In [ ]:
for target in ['churn_14d', 'churn_30d']:
    pivot = df[df['target'] == target].pivot_table(
        index='algorithm', columns='cleanup', values='test_auc_roc', aggfunc='mean'
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='viridis', ax=ax)
    ax.set_title(f'AUC promedio: algorithm × cleanup ({target})')
    plt.tight_layout()
    plt.savefig(cfg.STUDY_INFORMES / f'03_heatmap_alg_clean_{target}.png', dpi=100)
    plt.show()

## 5. AUC por cutoff × spike (mejor algorithm × cleanup)

In [ ]:
for target in ['churn_14d', 'churn_30d']:
    sub = df[df['target'] == target]
    # Encontrar (alg, cleanup) con mayor AUC mediano
    best_combo = sub.groupby(['algorithm', 'cleanup'])['test_auc_roc'].mean().idxmax()
    print(f'{target}: mejor combo (alg, cleanup) = {best_combo}')

    sub_best = sub[(sub['algorithm'] == best_combo[0]) & (sub['cleanup'] == best_combo[1])]
    pivot = sub_best.pivot_table(
        index='cutoff', columns='spike', values='test_auc_roc', aggfunc='mean'
    )
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='viridis', ax=ax)
    ax.set_title(f'AUC: cutoff × spike (fijo: {best_combo[0]} + {best_combo[1]}) — {target}')
    plt.tight_layout()
    plt.savefig(cfg.STUDY_INFORMES / f'03_heatmap_cutoff_spike_{target}.png', dpi=100)
    plt.show()

## 6. Overfitting gap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, target in zip(axes, ['churn_14d', 'churn_30d']):
    sub = df[df['target'] == target]
    sns.boxplot(data=sub, x='algorithm', y='overfitting_gap', hue='cleanup', ax=ax)
    ax.set_title(f'Overfitting gap por algorithm × cleanup ({target})')
    ax.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(cfg.STUDY_INFORMES / '03_overfitting_gap.png', dpi=100)
plt.show()

## 7. Informe ejecutivo

In [ ]:
report_path = cfg.STUDY_INFORMES / '03_screening_report.md'

lines = [f'# Screening Results — Sub-sesion 2+3\n']
lines.append(f'**Fecha**: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}\n')
lines.append(f'**Total modelos entrenados**: {len(df)}\n')
lines.append(f'**Configs**: {df["config_id"].nunique()}\n')
lines.append(f'**Tiempo total cómputo**: {df["training_time_s"].sum()/3600:.2f}h\n\n')

for target, baseline in [('churn_14d', BASELINE_14), ('churn_30d', BASELINE_30)]:
    sub = df[df['target'] == target].copy()
    sub['delta_vs_baseline'] = sub['test_auc_roc'] - baseline
    n_better = (sub['delta_vs_baseline'] > 0.001).sum()

    lines.append(f'## {target}\n')
    lines.append(f'- Baseline: {baseline:.4f}\n')
    lines.append(f'- Modelos que baten baseline: {n_better}/{len(sub)}\n\n')
    lines.append(f'### Top 10\n\n')
    top = sub.nlargest(10, 'test_auc_roc')[
        ['config_id', 'cutoff', 'spike', 'min_logins', 'cleanup', 'algorithm',
         'test_auc_roc', 'test_auc_pr', 'overfitting_gap']
    ]
    lines.append(top.to_markdown(index=False))
    lines.append('\n\n')

report_path.write_text(''.join(lines))
print(f'Reporte: {report_path}')